# B2: Evaluation

**Goal.** Evaluate all methods under the same rules on the 50 shared TriviaQA questions (Qwen2.5-0.5B): AUROC, AURC with randomly broken ties, and a bootstrap interval for the AUROC.

In [1]:
import sys

sys.path.insert(0, "..")  # so that `common` can be imported from notebooks/

import numpy as np
import pandas as pd

from common.data import load_data
from common.evaluate import evaluate
from common.summary import summary_table

rows = load_data("qwen0.5b", 50)
correct = np.array([row["correct"] for row in rows])
print("questions:", len(rows), "| correct:", int(correct.sum()), "| wrong:", int((~correct).sum()))


questions: 50 | correct: 12 | wrong: 38


## Implementation

`common/io.py` checks every score file against `CONTRACT.md`. `common/evaluate.py` computes AUROC and AURC (probly's `selective_prediction`, ties broken randomly over 20 shuffles) and a bootstrap interval. First check: fake scores must behave as expected.


In [2]:
fake = {
    "perfect": (~correct).astype(float),
    "random": np.random.default_rng(1).random(len(correct)),
    "constant": np.zeros(len(correct)),
    "inverted": correct.astype(float),
}
pd.DataFrame({name: evaluate(scores, correct) for name, scores in fake.items()}).T.round(3)

,auroc,aurc
perfect,1.000,0.466
random,0.526,0.758
constant,0.500,0.753
inverted,0.000,0.964


## Results


In [3]:
table = summary_table(rows)
table.round(3)

,method,status,auroc,ci_low,ci_high,aurc,n_failed,seconds_per_question
0,likelihood_mean,ok,0.814,0.656,0.943,0.605,0.0,0.000
1,likelihood_sum,ok,0.662,0.450,0.848,0.664,0.0,0.000
2,p_true_zeroshot_norm,ok,0.654,0.449,0.834,0.672,0.0,0.786
3,p_true_brainstorm,pending: no file yet,NaN,NaN,NaN,NaN,NaN,NaN
4,p_true_empirical,pending: no file yet,NaN,NaN,NaN,NaN,NaN,NaN
5,verbalized_2s,pending: no file yet,NaN,NaN,NaN,NaN,NaN,NaN
6,semantic_entropy,pending: no file yet,NaN,NaN,NaN,NaN,NaN,NaN
7,discrete_se,pending: no file yet,NaN,NaN,NaN,NaN,NaN,NaN


## Sanity test (R1, R2): 10 questions, correct vs wrong answer, 20 examples

In [4]:
from common.summary import sanity_table

sanity = sanity_table()
sanity.pivot(index="method", columns="model", values="auroc").round(2)

model,Qwen2.5-0.5B-Instruct,Qwen2.5-1.5B-Instruct,SmolLM2-135M-Instruct
method,,,
p_true_zeroshot_norm,0.70,0.97,0.40
p_true_zeroshot_raw,0.57,0.91,0.46
r2_letter_brainstorm,1.00,NaN,NaN
r2_letter_nobrainstorm,0.76,NaN,NaN


Note: the sanity AUROC treats the correct answer as the positive class and the confidence as the score, which is the reverse of the main table (wrong answer positive, uncertainty as score). There are only 20 examples, so the value moves in steps of about 0.05 and the results are very noisy.